# Imports

In [110]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [111]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from statsmodels.tsa.stattools import adfuller
from statsmodels.tools import add_constant
import plotly.graph_objects as go
from statsmodels.regression.linear_model import OLS
from statsmodels.tsa.vector_ar.vecm import coint_johansen
import itertools

# Constants

## Control

In [112]:
FETCH_DATA = False
SAVE_RESULT = True
PLOT_RESULT = False
PLOT_SPREAD = False

## Dataset

In [113]:
START_DATE = "2024-12-01"
END_DATE = "2025-12-01"

TOP_50 = ["BTC", "ETH", "BNB", "XRP", "SOL", "TRX", "DOGE", "ADA", "BCH", "LINK", "RAIN", "XMR", "XLM", "ZEC", "LEO", "LTC", "DAI", "SUI", "AVAX", "HBAR", "SHIB", "NIGHT", "STETH", "CRO", "UNI", "DOT", "AAVE", "CC", "BGB", "ASTER", "PI", "ENA", "SKY", "KCS", "WLD", "ONDO", "KAS", "APT", "ARB", "ALGO", "FLR", "ATOM", "FIL", "QNT", "VET", "M", "CBBTC", "WBT"]

## Routes

In [114]:
TICKERS_DIR = "tickers"
PLOTS_DIR = "plots"

# Configuration

# Functions

In [115]:
def fetch_from_yfinance(ticker: str, route: str, start_date, end_date, fetch: bool=True):
    if fetch:
        s = ticker.upper().strip()
        yahoo_format = s if s.endswith("-USD") else f"{s}-USD"
        
        df1 = yf.download(
            tickers=yahoo_format,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        df2 = yf.download(
            tickers=ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False
        )
        
        if len(df1) > 0:
            df = df1
        if len(df2) > 0:
            df = df2
            
        df.to_csv(route)
        df = pd.read_csv(route, skiprows=[1, 2], header=0)
        df = df.rename(columns={'Price': 'Date'})
        df = df.set_index('Date')
        df.to_csv(route)

    else:
        df = pd.read_csv(route)
    return df

In [116]:
def adf_test(ticker, df, column='Close'):
    if df is None or df.empty or column not in df.columns:
        return {"Ticker": ticker, "Error": "Missing Data"}

    series = df[column].dropna()
    
    if len(series) < 20:
        return {"Ticker": ticker, "Error": f"Insufficient data points: {len(series)}"}

    result = adfuller(series, autolag='AIC')
    
    adf_output = {
        "Ticker": ticker,
        "ADF Statistic": round(result[0], 4),
        "p-value": round(result[1], 4),
        "Stationary": result[1] < 0.05,
        "Lags Used": result[2],
        "Observations": result[3]
    }
    
    return adf_output

In [117]:
def hurst_exponent(series, q=2.0, max_lag=None, min_lag=2):
    series = np.asarray(series, dtype=float)
    n = len(series)

    if max_lag is None:
        max_lag = n // 4
    if max_lag <= min_lag:
        raise ValueError("max_lag must be > min_lag")

    # Lags
    lags = np.arange(min_lag, max_lag)

    # K_q values for each lag
    K = np.zeros_like(lags, dtype=float)

    for i, lag in enumerate(lags):
        diffs = np.abs(series[lag:] - series[:-lag])
        K[i] = np.mean(diffs ** q)

    # Fit log–log to estimate slope
    log_lags = np.log(lags)
    log_K = np.log(K)

    slope, _ = np.polyfit(log_lags, log_K, 1)

    Hurst = slope / q
    
    return Hurst

In [118]:
def half_life(series):
    series = series.dropna()
    
    if len(series) < 10:
        print("For half-life series must be longer than 10 values")
        return np.nan
    
    # Create lagged series
    price_lag = series.shift(1)
    price_diff = series - price_lag
    
    # Remove NaN values
    valid_data = pd.concat([price_lag, price_diff], axis=1).dropna()
    X = add_constant(valid_data.iloc[:, 0])
    y = valid_data.iloc[:, 1]
    
    # Perform OLS regression
    try:
        model = OLS(y, X)
        results = model.fit()
        
        beta = results.params.iloc[1]
        
        half_life = -np.log(2) / np.log(1 + beta) if (1 + beta) > 0 else np.nan
        
        return half_life
    except Exception as e:
        print(f"Error occured while calculating half life: ${e}")
        return np.nan

In [119]:
def store_series_plot(df, ticker: str, route: str = PLOTS_DIR):
    if not os.path.exists(route):
        os.makedirs(route)
        
    x_axis = df['Date'] if 'Date' in df.columns else df.index
        
    fig = go.Figure(data=[go.Candlestick(
        x=x_axis,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
    )])

    fig.update_layout(
        xaxis_title='Date',
        yaxis_title='Price',
        xaxis_rangeslider_visible=False,
        template='plotly_white',
        height=900,
        width=1600,
        showlegend=False
    )
    
    save_path = os.path.join(route, f"{ticker}.png")
    fig.write_image(save_path)

In [120]:
def johansen_test(df_combined, det_order=0, k_ar_diff=1):
    df_clean = df_combined.dropna()
    
    if df_clean.shape[1] < 2:
        print("Need at least 2 series for Johansen test.")
        return None

    result = coint_johansen(df_clean, det_order, k_ar_diff)
    
    traces = result.lr1
    cvts = result.cvt
    eigenvalues = result.eig
    
    summary = []
    for i in range(len(traces)):
        is_cointegrated = traces[i] > cvts[i, 1]
        summary.append({
            "Rank": i + 1,
            "Trace Stat": round(traces[i], 4),
            "CV 95%": round(cvts[i, 1], 4),
            "Significant": is_cointegrated
        })
    
    return pd.DataFrame(summary), result.evec[:, 0]

# Fetch data

## Fetch merket data of top 50 crypto

In [121]:
dfs = {}
if not os.path.exists(TICKERS_DIR):
    os.makedirs(TICKERS_DIR)
    print(f"Created directory: {TICKERS_DIR}")
for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        dfs[ticker] = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
        if PLOT_RESULT:
            store_series_plot(dfs[ticker], ticker)
    except Exception as e:
        print(f"Failed to fetch {ticker}: {e}")

# Calculate parameters

In [122]:
summary_results = []

for ticker in TOP_50:
    filename = os.path.join(TICKERS_DIR, f"{ticker}.csv")
    try:
        df = fetch_from_yfinance(ticker, filename, START_DATE, END_DATE, FETCH_DATA)
        res = adf_test(ticker, df)
        res['hurst'] = hurst_exponent(df['Close'])
        res['half_life'] = half_life(df['Close'])
        summary_results.append(res)
        
    except Exception as e:
        print(f"Failed {ticker}: {e}")

Failed XRP: max_lag must be > min_lag


In [123]:
summary_df = pd.DataFrame(summary_results)
summary_df.head()

,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,hurst,half_life
0,BTC,-1.7554,0.4028,False,0,248,0.447950,26.127331
1,ETH,-1.2146,0.6673,False,1,247,0.611152,47.294757
2,BNB,-1.0345,0.7405,False,8,356,0.466865,64.483378
3,SOL,-2.1083,0.2412,False,1,247,0.485505,14.621300
4,TRX,-0.1726,0.9417,False,12,236,0.538031,1693.142597


## Filter tickers with p-value $\le$ 0.05

In [124]:
stationary_df = summary_df[summary_df['p-value'] <= 0.05].copy()
stationary_df = stationary_df.sort_values(by='p-value')
print("Stationary tickers:", len(stationary_df))
stationary_df

Stationary tickers: 10


,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,hurst,half_life
15,DAI,-5.8488,0.0000,True,3,361,0.017397,0.698316
24,DOT,-4.6399,0.0001,True,6,358,0.461340,33.120561
16,SUI,-4.5019,0.0002,True,0,248,0.199470,4.157290
14,LTC,-4.1974,0.0007,True,0,248,0.143460,5.701002
12,ZEC,-3.7421,0.0036,True,13,351,0.561369,2855.264764
41,FIL,-3.7385,0.0036,True,4,360,0.448877,32.888921
20,NIGHT,-3.5955,0.0058,True,16,299,0.450949,30.989480
33,WLD,-3.3193,0.0140,True,2,362,0.481056,24.148911
11,XLM,-3.1492,0.0231,True,0,364,0.410398,18.637805
19,SHIB,-3.0610,0.0296,True,2,362,0.497374,24.642140


In [125]:
half_life_df = stationary_df.sort_values(by='half_life')
half_life_df

,Ticker,ADF Statistic,p-value,Stationary,Lags Used,Observations,hurst,half_life
15,DAI,-5.8488,0.0000,True,3,361,0.017397,0.698316
16,SUI,-4.5019,0.0002,True,0,248,0.199470,4.157290
14,LTC,-4.1974,0.0007,True,0,248,0.143460,5.701002
11,XLM,-3.1492,0.0231,True,0,364,0.410398,18.637805
33,WLD,-3.3193,0.0140,True,2,362,0.481056,24.148911
19,SHIB,-3.0610,0.0296,True,2,362,0.497374,24.642140
20,NIGHT,-3.5955,0.0058,True,16,299,0.450949,30.989480
41,FIL,-3.7385,0.0036,True,4,360,0.448877,32.888921
24,DOT,-4.6399,0.0001,True,6,358,0.461340,33.120561
12,ZEC,-3.7421,0.0036,True,13,351,0.561369,2855.264764


# Mix dfs

In [126]:
found_combinations = []
non_stationary_tickers = summary_df[summary_df['p-value'] > 0.05]['Ticker'].tolist()

for r in [4, 5]:
    if len(found_combinations) >= 10: break
    
    potential_groups = list(itertools.combinations(non_stationary_tickers[:20], r)) 

    for group in potential_groups:
        if len(found_combinations) >= 10: break
        
        combined_data = pd.concat([dfs[t]['Close'] for t in group], axis=1).dropna()
        
        res = coint_johansen(combined_data, det_order=0, k_ar_diff=1)
        
        if res.lr1[0] > res.cvt[0, 1]: 
            weights = res.evec[:, 0] 
            found_combinations.append({
                'group': group,
                'weights': weights,
                'johansen_stat': res.lr1[0]
            })

In [127]:
results_with_hl = []

for item in found_combinations:
    group = item['group']
    weights = item['weights']
    
    # Calculate the Spread series
    spread_series = pd.Series(0, index=dfs[group[0]].index)
    for i, ticker in enumerate(group):
        spread_series += dfs[ticker]['Close'] * weights[i]
    
    # Calculate Half-Life using your previously defined function
    hl = half_life(spread_series)
    
    results_with_hl.append({
        'group': group,
        'half_life': hl,
        'spread_series': spread_series,
        'weights': weights
    })

# Sort by half-life and pick the top 5
top_5_combinations = sorted(results_with_hl, key=lambda x: x['half_life'])[:5]

In [128]:
def plot_spread_candlestick(group, weights, dfs, folder="plots"):
    spread_df = pd.DataFrame(index=dfs[group[0]].index)
    
    # Close and Open are straightforward linear combinations
    spread_df['Close'] = sum(dfs[ticker]['Close'] * weights[i] for i, ticker in enumerate(group))
    spread_df['Open']  = sum(dfs[ticker]['Open'] * weights[i] for i, ticker in enumerate(group))
    
    # For High and Low, we must check the sign of the weight
    spread_high = 0
    spread_low = 0
    for i, ticker in enumerate(group):
        if weights[i] >= 0:
            spread_high += dfs[ticker]['High'] * weights[i]
            spread_low  += dfs[ticker]['Low'] * weights[i]
        else:
            # If weight is negative, the asset's 'Low' contributes to the spread's 'High'
            spread_high += dfs[ticker]['Low'] * weights[i]
            spread_low  += dfs[ticker]['High'] * weights[i]
            
    spread_df['High'] = spread_high
    spread_df['Low']  = spread_low
    
    # Store the result
    ticker_name = f"Spread_{'_'.join(group)}"
    store_series_plot(spread_df, ticker_name, folder)

In [129]:
import plotly.graph_objects as go
import os

def plot_overlay_normalized_candlesticks(group, dfs, ticker_name, folder="plots"):
    """
    Plots multiple normalized candlestick series on a single shared axis.
    """
    fig = go.Figure()

    # Distinct color pairs for up/down candles per ticker
    color_pairs = [
        {'inc': '#26a69a', 'dec': '#ef5350'}, # Teal / Red
        {'inc': '#2196f3', 'dec': '#f44336'}, # Blue / Bright Red
        {'inc': '#9c27b0', 'dec': '#e91e63'}, # Purple / Pink
        {'inc': '#ff9800', 'dec': '#795548'}, # Orange / Brown
        {'inc': '#4caf50', 'dec': '#ffeb3b'}  # Green / Yellow
    ]

    for i, ticker in enumerate(group):
        if ticker not in dfs:
            continue
            
        df = dfs[ticker].copy()
        
        # 1. Normalization: Divide all OHLC columns by the first Close price
        base_price = df['Close'].iloc[0]
        df_norm = df[['Open', 'High', 'Low', 'Close']] / base_price
        
        # 2. Select colors for this specific ticker
        colors = color_pairs[i % len(color_pairs)]
        
        # 3. Add the trace
        fig.add_trace(go.Candlestick(
            x=df.index,
            open=df_norm['Open'],
            high=df_norm['High'],
            low=df_norm['Low'],
            close=df_norm['Close'],
            name=f"{ticker} (Norm)",
            increasing_line_color=colors['inc'],
            decreasing_line_color=colors['dec'],
            increasing_fillcolor=colors['inc'],
            decreasing_fillcolor=colors['dec']
        ))

    # 4. Final Layout Adjustments
    fig.update_layout(
        title=f"Overlay Normalized Candlesticks: {', '.join(group)}",
        xaxis_title="Date",
        yaxis_title="Normalized Price (Base = 1.0)",
        xaxis_rangeslider_visible=False,
        template="plotly_white",
        height=600,
        width=1000,
        # This helps see the different candles if they overlap heavily
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )

    if not os.path.exists(folder):
        os.makedirs(folder)
        
    save_path = os.path.join(folder, f"Overlay_Normalized_{ticker_name}.png")
    fig.write_image(save_path)
    print(f"Overlay plot saved to {save_path}")

In [130]:
def plot_spread_candlestick(group, weights, dfs, folder="plots"):
    """
    محاسبه و رسم نمودار شمعی برای ترکیب خطی (Spread) چند رمزارز
    """
    # ایجاد یک دیتافریم خالی با ایندکس زمانی مشترک
    spread_df = pd.DataFrame(index=dfs[group[0]].index)
    
    # محاسبه Open و Close (ترکیب خطی ساده)
    spread_df['Open'] = sum(dfs[ticker]['Open'] * weights[i] for i, ticker in enumerate(group))
    spread_df['Close'] = sum(dfs[ticker]['Close'] * weights[i] for i, ticker in enumerate(group))
    
    # محاسبه High و Low با در نظر گرفتن علامت وزن‌ها
    spread_high = 0
    spread_low = 0
    for i, ticker in enumerate(group):
        if weights[i] >= 0:
            # وزن مثبت: High همان High است
            spread_high += dfs[ticker]['High'] * weights[i]
            spread_low  += dfs[ticker]['Low'] * weights[i]
        else:
            # وزن منفی: High به Low تبدیل می‌شود و بالعکس
            spread_high += dfs[ticker]['Low'] * weights[i]
            spread_low  += dfs[ticker]['High'] * weights[i]
            
    spread_df['High'] = spread_high
    spread_df['Low']  = spread_low

    # رسم نمودار با استفاده از Plotly
    ticker_name = "_".join(group)
    fig = go.Figure(data=[go.Candlestick(
        x=spread_df.index,
        open=spread_df['Open'],
        high=spread_df['High'],
        low=spread_df['Low'],
        close=spread_df['Close'],
        increasing_line_color='#26a69a', # رنگ سبز برای صعودی
        decreasing_line_color='#ef5350'  # رنگ قرمز برای نزولی
    )])

    fig.update_layout(
        title=f"Stationary Spread Candlestick (Group: {ticker_name})",
        xaxis_title="Date",
        yaxis_title="Spread Value",
        xaxis_rangeslider_visible=False,
        template="plotly_white",
        height=600
    )

    if not os.path.exists(folder):
        os.makedirs(folder)
        
    save_path = os.path.join(folder, f"Spread_Candle_{ticker_name}.png")
    fig.write_image(save_path)
    print(f"Spread candlestick saved to {save_path}")


In [131]:
if PLOT_SPREAD:
    for comb in top_5_combinations:
        group = comb['group']
        weights = comb['weights']
        ticker_name = "_".join(group)
        
        # Plot all 3-5 cryptos on the SAME chart
        plot_overlay_normalized_candlesticks(group, dfs, ticker_name)
        
        # Plot the resulting Spread Candlestick (as created previously)
        plot_spread_candlestick(group, weights, dfs)

# Strategy

In [132]:
list(map(lambda comb: comb['group'], top_5_combinations))

[('BTC', 'ETH', 'BNB', 'RAIN'),
 ('BTC', 'ETH', 'BNB', 'SOL'),
 ('BTC', 'ETH', 'BNB', 'AAVE'),
 ('BTC', 'ETH', 'SOL', 'STETH'),
 ('BTC', 'ETH', 'BNB', 'LINK')]

In [133]:
# for comb in top_5_combinations:
#     comb['Return'] = comb['spread_series'].pct_change() * 100

In [134]:
def calculate_strategy_params(spread_df, hl_value):
    lookback = int(hl_value * (int(np.ceil(30 / hl_value)) if hl_value < 30 else 2))
    
    ema_range = spread_df['Close'].ewm(span=lookback, adjust=False).mean()
    std_spread = spread_df['Close'].rolling(window=lookback).std()
    
    return lookback, ema_range, std_spread

In [135]:
strategy_results = []

for comb in top_5_combinations:
    group = comb['group']
    weights = comb['weights']
    hl = comb['half_life']
    
    spread_df = pd.DataFrame(index=dfs[group[0]].index)
    spread_df['Close'] = sum(dfs[ticker]['Close'] * weights[i] for i, ticker in enumerate(group))
    spread_df['Open']  = sum(dfs[ticker]['Open'] * weights[i] for i, ticker in enumerate(group))
    
    s_high, s_low = 0, 0
    for i, ticker in enumerate(group):
        if weights[i] >= 0:
            s_high += dfs[ticker]['High'] * weights[i]
            s_low  += dfs[ticker]['Low'] * weights[i]
        else:
            s_high += dfs[ticker]['Low'] * weights[i]
            s_low  += dfs[ticker]['High'] * weights[i]
    spread_df['High'], spread_df['Low'] = s_high, s_low
    
    lb, ema, std = calculate_strategy_params(spread_df, hl)
    
    spread_df['std'] = std
    spread_df['ema'] = ema
    
    print(f"\nGroup: {'-'.join(group)}")
    print(f"Half-life: {hl:.2f} days")
    print(f"Calculated Lookback: {lb} days")
    print(f"Latest EMA (of Range): {ema.iloc[-1]:.4f}")
    print(f"Latest STD: {std.iloc[-1]:.4f}")
    
    strategy_results.append({
        'group': group,
        'spread_df': spread_df,
        'ema_range': ema,
        'std_spread': std,
        'lookback': lb
    })


Group: BTC-ETH-BNB-RAIN
Half-life: 3.54 days
Calculated Lookback: 31 days
Latest EMA (of Range): 33.4847
Latest STD: nan

Group: BTC-ETH-BNB-SOL
Half-life: 4.16 days
Calculated Lookback: 33 days
Latest EMA (of Range): 21.7989
Latest STD: 0.9302

Group: BTC-ETH-BNB-AAVE
Half-life: 5.24 days
Calculated Lookback: 31 days
Latest EMA (of Range): 25.2697
Latest STD: 0.9786

Group: BTC-ETH-SOL-STETH
Half-life: 5.49 days
Calculated Lookback: 32 days
Latest EMA (of Range): 18.3218
Latest STD: 1.1045

Group: BTC-ETH-BNB-LINK
Half-life: 6.07 days
Calculated Lookback: 30 days
Latest EMA (of Range): 26.1326
Latest STD: 0.9649


## short signal

In [ ]:
for result in strategy_results:
    result['short_signal1'] = np.where(df['High'] > df['ema'] + df['std'], 1, 0)
    result['short_signal2'] = np.where(df['High'] > df['ema'] + 2 * df['std'], 1, 0)

,Close,Open,High,Low,std,ema
0,31.968343,32.994920,33.937362,30.422756,NaN,31.968343
1,31.192921,31.409919,32.495261,29.991621,NaN,31.919879
2,33.813748,30.774982,34.698943,29.027104,NaN,32.038246
3,34.013667,34.379229,36.905150,32.874990,NaN,32.161709
4,32.940741,33.656190,34.372061,31.569609,NaN,32.210399
...,...,...,...,...,...,...
244,NaN,NaN,NaN,NaN,NaN,33.484667
245,NaN,NaN,NaN,NaN,NaN,33.484667
246,NaN,NaN,NaN,NaN,NaN,33.484667
247,NaN,NaN,NaN,NaN,NaN,33.484667
